# Embeddings And Language Modeling

Token IDs are discrete addresses, not useful vectors. An embedding layer turns those addresses into learned coordinates, and a language-model head turns those coordinates back into logits over the vocabulary.


## Problem: What Must This Component Compute?

At this point in the model, the input is a tensor of token IDs with shape `(B, T)`. The component must map each ID to a dense vector, preserve token order for the later transformer, and produce logits that can score the next token.

We will first write the lookup directly with tensors so every dimension is visible. Only after the numerical checks pass will we wrap the operation in a reusable module.

Why this matters later: when an architecture paper claims to improve attention, memory, quantization, or world modeling, it is usually changing one of these constraints: what information can move across positions, how much memory is required, what objective is optimized, or which representations are preserved.


## One-Hot Vectors Versus Learned Lookup Matrices

If the vocabulary size is `V` and embedding width is `C`, an embedding matrix has shape `(V, C)`. Selecting token `i` is equivalent to multiplying a one-hot row vector `e_i` by that matrix:

\[
x_i = e_i W_E.
\]

`nn.Embedding` implements the same operation more efficiently than constructing one-hot tensors explicitly, but the math is identical.


In [ ]:
import torch

vocab_size = 10
n_embd = 4
embedding = torch.nn.Embedding(vocab_size, n_embd)
ids = torch.tensor([[1, 2, 3, 4]])
x = embedding(ids)
x.shape


In [ ]:
one_hot = torch.nn.functional.one_hot(ids, num_classes=vocab_size).float()
manual_lookup = one_hot @ embedding.weight

assert manual_lookup.shape == x.shape
assert torch.allclose(manual_lookup, x)

manual_lookup[0, 0], x[0, 0]


## Logits, Label Shifting, And Next-Token Loss

A language model predicts `x_{t+1}` from representations built from `x_{\le t}`. That means the training target is the input sequence shifted one step to the left.

If embeddings have shape `(B, T, C)` and the output projection has shape `(C, V)`, then logits have shape `(B, T, V)`. For next-token training we compare `logits[:, :-1, :]` against `ids[:, 1:]`.


In [ ]:
from llm_from_scratch.functional import cross_entropy_from_logits
import torch.nn.functional as F

torch.manual_seed(0)
projection = torch.randn(n_embd, vocab_size)
logits = x @ projection
shifted_logits = logits[:, :-1, :]
targets = ids[:, 1:]

loss = cross_entropy_from_logits(shifted_logits, targets)
reference = F.cross_entropy(shifted_logits.reshape(-1, vocab_size), targets.reshape(-1))

assert shifted_logits.shape == (1, 3, vocab_size)
assert targets.shape == (1, 3)
assert torch.allclose(loss, reference)

{"loss": loss.item(), "flat_logits_shape": shifted_logits.reshape(-1, vocab_size).shape}


## Loss Sanity Checks And Module Bridge

If every class receives the same logit, the model is maximally uncertain and the cross-entropy should become `log(V)`. This gives a quick check that the vocabulary dimension is wired correctly.

The full project wraps these pieces in `TinyGPT`, where token embeddings, position embeddings, transformer blocks, and the output head become a single `nn.Module`.


In [ ]:
from llm_from_scratch.configs import ModelConfig
from llm_from_scratch.model import TinyGPT

uniform_logits = torch.zeros(1, 3, vocab_size)
uniform_targets = torch.tensor([[1, 2, 3]])
uniform_loss = cross_entropy_from_logits(uniform_logits, uniform_targets)
assert torch.isclose(uniform_loss, torch.log(torch.tensor(float(vocab_size))))

cfg = ModelConfig(vocab_size=vocab_size, block_size=8, n_embd=n_embd, n_head=2, n_layer=1)
model = TinyGPT(cfg)
model_logits, model_loss = model(ids, torch.tensor([[2, 3, 4, 5]]))

assert model_logits.shape == (1, 4, vocab_size)
assert model.lm_head.weight.data_ptr() == model.token_embedding.weight.data_ptr()

{"uniform_loss": uniform_loss.item(), "model_loss": float(model_loss)}


## Exercise Checkpoint 1

1. Why is `nn.Embedding` equivalent to selecting rows from a matrix rather than multiplying by arbitrary dense inputs?
2. If `ids` has shape `(B, T)`, why do the targets for next-token prediction usually have shape `(B, T - 1)` in the simplest shifted setup?


## Exercise Checkpoint 2

1. What cross-entropy value do you expect from a vocabulary of size `V` if the logits are all zeros, and why?
2. Explain why tying the output head weights to the token embedding weights can save parameters while keeping the same vocabulary geometry.
